In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset#datasets library from hugging face
from tokenizers import Tokenizer, models,trainers, pre_tokenizers#Hugging face tokenizers library
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import math

In [ ]:
device="cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
vocab_size=10000
batch_size=32
window=32
emb_dim=128
hidden_dim=128
epochs=3

print(f"Using device:{device}")

Using device:cuda


In [ ]:
#Load WikiText-2
dataset=load_dataset("wikitext","wikitext-2-raw-v1",split="train")
print(len(dataset['text']))
text_data=[line for line in dataset['text']]
print(text_data[10])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

36718
 The game 's battle system , the BliTZ system , is carried over directly from Valkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters ' turns . Each character has a field and distance of movement limited by their Action Gauge . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked out by enemy attacks . Each character has specific " Potentials " , skills unique to each character . They are divided into " Personal Potential " , which are innate skills that remain unaltered unless otherwise dictated by the story and can either help or impede a cha

In [ ]:
#Tokenizing
tokenizer=Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer=pre_tokenizers.Whitespace()
trainer=trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=["[UNK]","[PAD]","[BOS]","[EOS]"])
tokenizer.train_from_iterator(text_data,trainer)

#Tokenize entire dataset into one long sequence
all_tokens=[]
for line in text_data:
    all_tokens.extend(tokenizer.encode(line).ids)
print(all_tokens[0:10])

[32, 8853, 9637, 3106, 32, 54, 3524, 185, 1305, 8853]


In [ ]:
#Creating Pytorch dataset for self-supervised learning
#Takes tokens and window size and generates dataset for Pytorch model
class TextDataset(Dataset):
    def __init__(self, tokens,seq_len):
        self.tokens=torch.tensor(tokens,dtype=torch.long)
        self.seq_len=seq_len#window size

    def __len__(self):
        return len(self.tokens)-self.seq_len-1

    def __getitem__(self,idx):
        #Input: tokens[0:32], Target: tokens[1:33]
        #RNN having output from each cell (Many to Many)
        #This is better learning
        x=self.tokens[idx:idx+self.seq_len]
        y=self.tokens[idx+1:idx+self.seq_len+1]
        return x,y
dataset=TextDataset(all_tokens,window)
#Dataset in the format required by Pytorch
loader=DataLoader(dataset,batch_size=batch_size,shuffle=True)

In [ ]:
#LSTM model
class myRNN(nn.Module):
    def __init__(self, vocab_size,embed_dim,hidden_dim):
        super().__init__()
        self.embedding=nn.Embedding(num_embeddings=vocab_size,embedding_dim=embed_dim)
        #nn.Embedding is essentially a look-up table
        self.lstm=nn.LSTM(input_size=embed_dim,hidden_size=hidden_dim,num_layers=2,batch_first=True,dropout=0.2)
        #batch_first=True; Input and Output tensors in following format (batch, seq, feature)
        self.fc=nn.Linear(in_features=hidden_dim,out_features=vocab_size)

    def forward(self,x,hidden=None):
        x=self.embedding(x)
        #hidden is actually a tuple in lstm; hidden[0] is h[t-1] and hidden[1] is c[t-1]
        out,hidden=self.lstm(x,hidden)
        y=self.fc(out)
        return y,hidden

mymod=myRNN(vocab_size=vocab_size,embed_dim=emb_dim,hidden_dim=emb_dim).to(device)
optimizer=torch.optim.Adam(params=mymod.parameters(),lr=1e-3)
criterion=nn.CrossEntropyLoss()

In [ ]:
#Training Loop
for e in range(epochs):
    mymod.train()
    total_loss=0
    for batch_idx,(x,y) in enumerate(loader):
        x,y=x.to(device),y.to(device)#Inputs and outputs
        optimizer.zero_grad()
        #forward pass
        out,_=mymod(x)
        loss=criterion(out.view(-1,vocab_size),y.view(-1))#Evaluated in batches
        #Backpropagation
        loss.backward()#Calculate gradients
        optimizer.step()#Update weights

        total_loss+=loss.item()
        if batch_idx % 500==0:
            print(f"Epoch {e+1} | Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.2f}")
    avg_loss=total_loss/len(loader)
    print(f"Epoch {e+1} Avg. Loss: {avg_loss:.2f} | Perplexity :{math.exp(avg_loss):.2f}")

Epoch 1 | Batch 0/79650 | Loss: 9.22
Epoch 1 | Batch 500/79650 | Loss: 7.04
Epoch 1 | Batch 1000/79650 | Loss: 6.72
Epoch 1 | Batch 1500/79650 | Loss: 6.27
Epoch 1 | Batch 2000/79650 | Loss: 6.32
Epoch 1 | Batch 2500/79650 | Loss: 6.19
Epoch 1 | Batch 3000/79650 | Loss: 6.37
Epoch 1 | Batch 3500/79650 | Loss: 6.11
Epoch 1 | Batch 4000/79650 | Loss: 6.02
Epoch 1 | Batch 4500/79650 | Loss: 5.92
Epoch 1 | Batch 5000/79650 | Loss: 5.83
Epoch 1 | Batch 5500/79650 | Loss: 5.84
Epoch 1 | Batch 6000/79650 | Loss: 5.78
Epoch 1 | Batch 6500/79650 | Loss: 5.85
Epoch 1 | Batch 7000/79650 | Loss: 5.61
Epoch 1 | Batch 7500/79650 | Loss: 5.77
Epoch 1 | Batch 8000/79650 | Loss: 5.61
Epoch 1 | Batch 8500/79650 | Loss: 5.59
Epoch 1 | Batch 9000/79650 | Loss: 5.54
Epoch 1 | Batch 9500/79650 | Loss: 5.63
Epoch 1 | Batch 10000/79650 | Loss: 5.65
Epoch 1 | Batch 10500/79650 | Loss: 5.36
Epoch 1 | Batch 11000/79650 | Loss: 5.44
Epoch 1 | Batch 11500/79650 | Loss: 5.52
Epoch 1 | Batch 12000/79650 | Loss: 5.59

In [ ]:
from google.colab import files
torch.save(mymod.state_dict(),"model.pth")

In [ ]:
files.download("/content/model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def predict(text,mymod,tokenizer,top_k=5):
    mymod.eval()
    tokens=tokenizer.encode(text).ids
    #nn.Embedding takes care of one-hot encoding and multiplication with embedding matrix
    #tokens is 1D list; [tokens] is 2D list; Pytorch models work only on batches (by default)
    input_tensor=torch.tensor([tokens]).to(device)

    with torch.no_grad():
        y,_=mymod(input_tensor)
        last_token=y[0,-1,:]# batch, seq, features; 0 batch, last token (-1) and all features
        probs=torch.softmax(last_token,dim=-1)
        #if previous line is not present then last_token is usually obtained for batch
        #In that case softmax needs to be evaluated for last dimension
        top_probs,top_indices=torch.topk(probs,top_k)
        #Storing results as string in a list
        results=[]
        for i in range(top_k):
            token_str=tokenizer.decode([top_indices[i].item()])
            results.append((token_str,top_probs[i].item()))
    return(results)

inp=input("Enter string:")
predictions=predict(inp,mymod,tokenizer)
for word,prob in predictions:
    print(f"{word}({prob:.2f})")

Enter string:players select each
of(0.22)
other(0.18)
team(0.05)
side(0.03)
place(0.03)


In [ ]:
#Load file and run in evaluation
mymod.load_state_dict(torch.load("/content/textcompletionrnn.pth", map_location='cuda'))
def predict(text,mymod,tokenizer,top_k=5):
    mymod.eval()
    tokens=tokenizer.encode(text).ids
    #nn.Embedding takes care of one-hot encoding and multiplication with embedding matrix
    #tokens is 1D list; [tokens] is 2D list; Pytorch models work only on batches (by default)
    input_tensor=torch.tensor([tokens]).to(device)

    with torch.no_grad():
        y,_=mymod(input_tensor)
        last_token=y[0,-1,:]# batch, seq, features; 0 batch, last token (-1) and all features
        probs=torch.softmax(last_token,dim=-1)
        #if previous line is not present then last_token is usually obtained for batch
        #In that case softmax needs to be evaluated for last dimension
        top_probs,top_indices=torch.topk(probs,top_k)
        #Storing results as string in a list
        results=[]
        for i in range(top_k):
            token_str=tokenizer.decode([top_indices[i].item()])
            results.append((token_str,top_probs[i].item()))
    return(results)

inp=input("Enter string:")
predictions=predict(inp,mymod,tokenizer)
for word,prob in predictions:
    print(f"{word}({prob:.2f})")

Enter string:The capital of India is
the(0.09)
a(0.07)
not(0.04)
located(0.03)
also(0.03)
